# 🌩️ AegisGrid — XGBoost Disaster Prediction Pipeline (v3)
### End-to-End ML: Classification (Disaster Type) + Regression (Severity Magnitude)

**Pipeline Map:**
1. 📦 Load & Explore Data
2. 🔧 Feature Engineering — `disaster_type` label + `disaster_severity` target
3. 🧹 Preprocessing — OneHotEncode categoricals, drop ALL leakage columns
4. 🎯 Feature Selection (with variance filter)
5. ✂️ Train / Test Split (80/20 stratified)
6. ⚖️ Class Imbalance — `scale_pos_weight` / `class_weight` inside XGBoost (no SMOTE)
7. 🌲 XGBoost Classifier → *Flood | Water Clogging | Heatwave | Lightning*
8. 📈 XGBoost Regressor → *Severity score 0–100 (leakage-free)*
9. 📊 Evaluation — Accuracy, F1, RMSE, R², Confusion Matrix
10. 🔍 SHAP Feature Importance
11. 💾 Save Models as sklearn Pipeline

> **Install once:** `pip install xgboost shap`

> **All fixes applied in this version:**
> - ✅ Correct CSV filename
> - ✅ Complete leakage column list
> - ✅ Target-leakage probabilities excluded from features
> - ✅ `OneHotEncoder` for nominal categoricals (no `LabelEncoder` on features)
> - ✅ Binary `is_*` columns cast to int before OHE (no spurious categories)
> - ✅ `StandardScaler` removed (tree models are scale-invariant)
> - ✅ Real `xgboost` library used throughout
> - ✅ **Class imbalance handled via `sample_weight` (no SMOTE)**
> - ✅ Pipeline re-fit fixed — models trained inside pipeline on raw data (no double-transform)
> - ✅ SHAP explainer defined before it is used; `import shap` added
> - ✅ SHAP uses real feature names (not `f0, f1, f2...`)
> - ✅ Low-variance feature filter added
> - ✅ Full visualization cell (8 plots)


## Step 1 — 📦 Load & Explore Data

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('aegisgrid_final_dataset.csv')
print(f'Shape          : {df.shape}')
print(f'Missing values : {df.isnull().sum().sum()}')
df.head(3)


Shape          : (15000, 99)
Missing values : 0


,record_id,year,month,day_of_year,hour_of_day,is_monsoon_season,is_summer_season,is_pre_monsoon,is_afternoon_peak,sector_name,...,structures_at_lightning_risk,power_outage_probability_pct,casualties_risk_lightning,property_damage_lakh,rescue_operation_cost_lakh,relief_rehabilitation_cost_lakh,infrastructure_repair_cost_lakh,heat_relief_medical_cost_lakh,lightning_damage_cost_lakh,total_estimated_cost_lakh
0,1,2023,11,189,7,0,0,0,0,Rasulgarh,...,6,14.1,2,1.31,11.65,21.08,56.51,8.70,29.78,129.03
1,2,2020,4,222,3,0,1,1,0,Patia,...,12,37.7,3,4.55,8.42,20.98,28.65,12.97,40.14,115.71
2,3,2024,4,54,23,0,1,1,0,Old Town,...,9,37.6,2,1.02,15.16,31.82,17.70,18.04,27.61,111.35


In [4]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
record_id,15000.0,7500.50,4330.27,1.00,3750.75,7500.50,11250.25,15000.00
year,15000.0,2022.23,1.39,2020.00,2021.00,2022.00,2023.00,2024.00
month,15000.0,6.48,3.45,1.00,3.00,6.00,10.00,12.00
day_of_year,15000.0,182.29,105.76,1.00,90.00,182.00,274.00,365.00
hour_of_day,15000.0,11.51,6.92,0.00,5.00,12.00,18.00,23.00
...,...,...,...,...,...,...,...,...
relief_rehabilitation_cost_lakh,15000.0,34.31,17.87,5.62,20.36,29.36,45.17,120.92
infrastructure_repair_cost_lakh,15000.0,46.20,26.35,1.75,25.82,41.24,62.93,136.62
heat_relief_medical_cost_lakh,15000.0,10.23,3.47,3.36,7.64,9.39,12.31,27.06
lightning_damage_cost_lakh,15000.0,45.70,20.19,6.35,31.12,42.21,56.58,165.05


## Step 2 — 🔧 Feature Engineering

### 2a. `disaster_type` label (rule-based scoring)
| Disaster | Key Signals |
|---|---|
| **Lightning** | wind > 55 km/h + visibility < 4 km + rainfall |
| **Flood** | flood_probability > 0.60 + heavy rain + water level |
| **Water Clogging** | blocked drains + impervious surface + moderate rain |
| **Heatwave** | heat_probability > 0.60 + temp > 38 °C + solar radiation |

### 2b. `disaster_severity` regression target (0–100)
> `flood_probability` and `heat_probability` are used in this formula and are therefore
> excluded from features in Step 4 to prevent target leakage.


In [5]:
def assign_disaster_type(row):
    scores = {
        'Lightning':      (row['wind_speed_kmh'] > 55) * 2 +
                          (row['visibility_km'] < 4) +
                          (row['rainfall_mm_per_hr'] > 10),

        'Flood':          (row['flood_probability'] > 0.60) * 2 +
                          (row['rainfall_mm_per_hr'] > 30) +
                          (row['water_level_m'] > 3),

        'Water Clogging': (row['blocked_drains_count'] > 10) * 2 +
                          (row['impervious_surface_pct'] > 65) +
                          (row['rainfall_mm_per_hr'] > 15) +
                          (row['flood_probability'] > 0.35),

        'Heatwave':       (row['heat_probability'] > 0.60) * 2 +
                          (row['temperature_c'] > 38) +
                          (row['solar_radiation_wm2'] > 700),
    }
    return max(scores, key=scores.get)

df['disaster_type'] = df.apply(assign_disaster_type, axis=1)
print('Disaster Type Distribution:')
print(df['disaster_type'].value_counts())
print()
print('Percentage:')
print((df['disaster_type'].value_counts(normalize=True)*100).round(2))


Disaster Type Distribution:
disaster_type
Water Clogging    5176
Heatwave          3651
Lightning         3311
Flood             2862
Name: count, dtype: int64

Percentage:
disaster_type
Water Clogging    34.51
Heatwave          24.34
Lightning         22.07
Flood             19.08
Name: proportion, dtype: float64


In [6]:
# Composite severity score (0–100)
# flood_probability and heat_probability are used here intentionally as target signals.
# They are EXCLUDED from features in Step 4 to prevent leakage.
df['disaster_severity'] = (
    df['flood_probability']              * 20 +
    df['heat_probability']               * 20 +
    (df['temperature_c'] - 30).clip(0)  * 1.5 +
    df['rainfall_mm_per_hr'].clip(0,100) * 0.3 +
    df['wind_speed_kmh'].clip(0,80)      * 0.3 +
    df['blocked_drains_count']           * 0.5
).clip(0, 100)

print('Severity statistics:')
print(df['disaster_severity'].describe().round(3))


Severity statistics:
count    15000.000
mean        45.470
std         16.240
min         11.350
25%         32.706
50%         45.252
75%         56.004
max        100.000
Name: disaster_severity, dtype: float64


## Step 3 — 🧹 Preprocessing

In [7]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Complete leakage column list — all risk/prob/action derived columns
# AND the probability columns used in the severity formula
DROP_COLS = [
    'record_id',
    # Risk label/code columns (all derived — pure leakage)
    'flood_risk_level',    'flood_risk_code',
    'heat_risk_level',     'heat_risk_code',
    'lightning_risk_level','lightning_risk_code',
    'overall_disaster_risk','overall_risk_code',
    'recommended_action',
    # Probability columns used in the target formula
    'flood_probability',
    'heat_probability',
    'lightning_probability',
    # Other leakage-adjacent columns
    'power_outage_probability_pct',
    'casualties_risk_lightning',
    # Original string target (we use encoded version)
    'disaster_type',
]

# FIX: Cast binary is_* columns to int BEFORE detecting categoricals.
# Without this they are seen as object dtype on some pandas versions and get
# spurious OHE categories like "True" / "False" instead of 0/1.
binary_cols = ['is_monsoon_season','is_summer_season','is_pre_monsoon',
               'is_afternoon_peak','thunder_heard_30min','lightning_rods_installed']
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

# Identify nominal categorical columns (string dtype only, after binary cast)
cat_cols = [c for c in df.select_dtypes(include=['object']).columns
            if c not in ['disaster_type'] and c not in DROP_COLS]
print(f'Categorical columns to one-hot encode: {cat_cols}')

# Encode classification target
le_target = LabelEncoder()
df['disaster_type_enc'] = le_target.fit_transform(df['disaster_type'])
CLASS_NAMES = le_target.classes_
print(f'Class mapping: {dict(enumerate(CLASS_NAMES))}')


Categorical columns to one-hot encode: ['sector_name']
Class mapping: {0: 'Flood', 1: 'Heatwave', 2: 'Lightning', 3: 'Water Clogging'}


## Step 4 — 🎯 Feature Selection

In [8]:
from sklearn.feature_selection import VarianceThreshold

EXCLUDE = set(DROP_COLS + ['disaster_type_enc', 'disaster_severity'])
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE]

X     = df[FEATURE_COLS].copy()
y_clf = df['disaster_type_enc'].copy()
y_reg = df['disaster_severity'].copy()

# Identify numeric and categorical feature columns
cat_feature_cols = [c for c in FEATURE_COLS if c in cat_cols]
num_feature_cols = [c for c in FEATURE_COLS if c not in cat_cols]

# Low-variance filter on numeric columns only (removes near-constant features)
# Threshold=0.01 removes features where >99% of values are the same
vt = VarianceThreshold(threshold=0.01)
vt.fit(X[num_feature_cols])
low_var_mask = vt.get_support()
removed = [c for c, keep in zip(num_feature_cols, low_var_mask) if not keep]
num_feature_cols = [c for c, keep in zip(num_feature_cols, low_var_mask) if keep]
if removed:
    print(f'Low-variance features removed: {removed}')
else:
    print('No low-variance features removed.')

FEATURE_COLS = cat_feature_cols + num_feature_cols
X = df[FEATURE_COLS].copy()

print(f'\nTotal features    : {len(FEATURE_COLS)}')
print(f'  Numeric         : {len(num_feature_cols)}')
print(f'  Categorical     : {len(cat_feature_cols)} -> {cat_feature_cols}')
print(f'X shape           : {X.shape}')


Low-variance features removed: ['latitude', 'longitude']

Total features    : 82
  Numeric         : 81
  Categorical     : 1 -> ['sector_name']
X shape           : (15000, 82)


## Step 5 — ✂️ Train / Test Split (80/20 Stratified)

In [9]:
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

X_train, X_test, y_clf_train, y_clf_test, y_reg_train, y_reg_test = train_test_split(
    X, y_clf, y_reg,
    test_size=0.20,
    random_state=SEED,
    stratify=y_clf
)

print(f'Train : {X_train.shape[0]} samples')
print(f'Test  : {X_test.shape[0]} samples')
print('\nTrain class dist:')
print(pd.Series(y_clf_train).map(dict(enumerate(CLASS_NAMES))).value_counts())
print('\nTest class dist:')
print(pd.Series(y_clf_test).map(dict(enumerate(CLASS_NAMES))).value_counts())


Train : 12000 samples
Test  : 3000 samples

Train class dist:
disaster_type_enc
Water Clogging    4141
Heatwave          2921
Lightning         2649
Flood             2289
Name: count, dtype: int64

Test class dist:
disaster_type_enc
Water Clogging    1035
Heatwave           730
Lightning          662
Flood              573
Name: count, dtype: int64


## Step 6 — ⚖️ Class Imbalance Handling (no SMOTE)

The class distribution is moderately imbalanced (Water Clogging ~35%, Flood ~19%).

**Approach:** Pass `sample_weight` to `XGBClassifier.fit()`.
Each training sample is upweighted inversely proportional to its class frequency.
This is equivalent to `class_weight='balanced'` in sklearn but works natively with XGBoost.

**Why not SMOTE?**
- SMOTE synthesises new samples by interpolating between existing ones in feature space.
  For this dataset (mix of meteorological floats + infrastructure integers + binary flags),
  interpolated samples can produce physically impossible values (e.g. negative drain counts,
  fractional binary flags).
- `sample_weight` achieves the same loss-function correction with zero data fabrication.


In [10]:
from sklearn.utils.class_weight import compute_sample_weight

# Compute per-sample weights inversely proportional to class frequency
sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_clf_train)

print('Class weights summary:')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = (y_clf_train == cls_idx)
    w    = sample_weights_train[mask][0]
    print(f'  {cls_name:<18}: weight = {w:.4f}  (n = {mask.sum()})')


Class weights summary:
  Flood             : weight = 1.3106  (n = 2289)
  Heatwave          : weight = 1.0270  (n = 2921)
  Lightning         : weight = 1.1325  (n = 2649)
  Water Clogging    : weight = 0.7245  (n = 4141)


## Step 7 — 🔧 Preprocessing Pipeline (OneHotEncoder)

> `StandardScaler` is **not** used. XGBoost and gradient-boosted trees split on
> thresholds — scaling features has zero effect and only adds confusion.
> `OneHotEncoder` is applied to nominal string columns inside a `ColumnTransformer`.


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_feature_cols)
    ],
    remainder='passthrough'
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

# Build final feature name list — used by SHAP for axis labels
ohe_feature_names  = preprocessor.named_transformers_['cat'] \
                        .get_feature_names_out(cat_feature_cols).tolist()
all_feature_names  = ohe_feature_names + num_feature_cols

print(f'After encoding: {X_train_proc.shape[1]} features '
      f'({len(ohe_feature_names)} from OHE, {len(num_feature_cols)} numeric)')


After encoding: 86 features (5 from OHE, 81 numeric)


## Step 8 — 🌲 XGBoost Classifier
**Predicts: Flood | Water Clogging | Heatwave | Lightning**


In [12]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

clf = XGBClassifier(
    n_estimators      = 300,
    max_depth         = 6,
    learning_rate     = 0.10,
    subsample         = 0.80,
    colsample_bytree  = 0.80,
    eval_metric       = 'mlogloss',
    random_state      = SEED,
    n_jobs            = -1
)

print('Training XGBoost Classifier (with sample_weight for class balance)...')
# Pass sample weights — XGBoost adjusts loss contribution per sample
clf.fit(X_train_proc, y_clf_train, sample_weight=sample_weights_train)
print('Done!')


Training XGBoost Classifier (with sample_weight for class balance)...
Done!


In [13]:
y_clf_pred = clf.predict(X_test_proc)
clf_acc = accuracy_score(y_clf_test, y_clf_pred)
clf_f1  = f1_score(y_clf_test, y_clf_pred, average='weighted')

print(f'Accuracy  : {clf_acc:.4f}  ({clf_acc*100:.2f}%)')
print(f'F1 (wtd)  : {clf_f1:.4f}')
print()
print(classification_report(y_clf_test, y_clf_pred, target_names=CLASS_NAMES))


Accuracy  : 0.9560  (95.60%)
F1 (wtd)  : 0.9561

                precision    recall  f1-score   support

         Flood       0.93      0.95      0.94       573
      Heatwave       0.95      0.97      0.96       730
     Lightning       0.98      0.96      0.97       662
Water Clogging       0.96      0.95      0.95      1035

      accuracy                           0.96      3000
     macro avg       0.96      0.96      0.96      3000
  weighted avg       0.96      0.96      0.96      3000



In [14]:
# Step 1: Encode labels
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_clf_train_enc = le.fit_transform(y_clf_train)
y_clf_test_enc = le.transform(y_clf_test)

# Step 2: Cross Validation
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_scores = cross_val_score(
    clf,
    X_train_proc,
    y_clf_train_enc,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1
)

print(f'5-Fold CV F1: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
print(f'Per-fold: {cv_scores.round(4)}')

5-Fold CV F1: 0.9620 +/- 0.0014
Per-fold: [0.9629 0.9642 0.9609 0.9604 0.9617]


In [15]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

reg = XGBRegressor(
    n_estimators      = 300,
    max_depth         = 6,
    learning_rate     = 0.10,
    subsample         = 0.80,
    colsample_bytree  = 0.80,
    random_state      = SEED,
    n_jobs            = -1
)

print('Training XGBoost Regressor...')
reg.fit(X_train_proc, y_reg_train)
print('Done!')


Training XGBoost Regressor...
Done!


In [23]:
y_reg_pred = reg.predict(X_test_proc)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
mae  = mean_absolute_error(y_reg_test, y_reg_pred)
r2   = r2_score(y_reg_test, y_reg_pred)

print(y_reg_pred)
print(f'RMSE  : {rmse:.4f}')
print(f'MAE   : {mae:.4f}')
print(f'R2    : {r2:.4f}  ({r2*100:.2f}% variance explained)')
print()
print('Note: R² lower than the leaky version is expected and correct.')
print('The model now generalises to unseen data rather than memorising the formula.')


[74.45983  33.971992 47.57372  ... 83.90317  39.89386  46.866642]
RMSE  : 1.1151
MAE   : 0.8500
R2    : 0.9954  (99.54% variance explained)

Note: R² lower than the leaky version is expected and correct.
The model now generalises to unseen data rather than memorising the formula.


## Step 10 — 📊 Evaluation Plots

## Step 11 — 🔍 SHAP Feature Importance

## Step 12 — 💾 Save Models as sklearn Pipeline

In [20]:
import joblib, os
from sklearn.utils.class_weight import compute_sample_weight

SAVE_DIR = 'models'
os.makedirs(SAVE_DIR, exist_ok=True)

# FIX: Build pipelines and fit them on RAW X_train (not pre-transformed X_train_proc).
# The Pipeline applies the preprocessor internally, so there is no double-transform.
# The clf and reg objects trained above are re-created fresh inside the pipeline.

from xgboost import XGBClassifier, XGBRegressor

clf_fresh = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.10,
    subsample=0.80, colsample_bytree=0.80,
    eval_metric='mlogloss', random_state=SEED, n_jobs=-1
)
reg_fresh = XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.10,
    subsample=0.80, colsample_bytree=0.80,
    random_state=SEED, n_jobs=-1
)

clf_pipeline = Pipeline([
    ('preprocessor', ColumnTransformer(
        transformers=[('cat', OneHotEncoder(handle_unknown='ignore',
                                            sparse_output=False), cat_feature_cols)],
        remainder='passthrough'
    )),
    ('classifier', clf_fresh)
])

reg_pipeline = Pipeline([
    ('preprocessor', ColumnTransformer(
        transformers=[('cat', OneHotEncoder(handle_unknown='ignore',
                                            sparse_output=False), cat_feature_cols)],
        remainder='passthrough'
    )),
    ('regressor', reg_fresh)
])

# Compute weights for raw training labels (same formula as Step 6)
sw = compute_sample_weight(class_weight='balanced', y=y_clf_train)

print('Fitting classifier pipeline on raw training data...')
clf_pipeline.fit(X_train, y_clf_train,
                 classifier__sample_weight=sw)

print('Fitting regressor pipeline on raw training data...')
reg_pipeline.fit(X_train, y_reg_train)

# Save
joblib.dump(clf_pipeline, f'{SAVE_DIR}/xgb_classifier_pipeline.pkl')
joblib.dump(reg_pipeline, f'{SAVE_DIR}/xgb_regressor_pipeline.pkl')
joblib.dump(le_target,    f'{SAVE_DIR}/label_encoder.pkl')

print('\nSaved:')
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1024
    print(f'  {SAVE_DIR}/{f}  ({size:.1f} KB)')


Fitting classifier pipeline on raw training data...
Fitting regressor pipeline on raw training data...

Saved:
  models/label_encoder.pkl  (0.5 KB)
  models/xgb_classifier_pipeline.pkl  (2634.8 KB)
  models/xgb_regressor_pipeline.pkl  (1328.0 KB)


## 🚀 Demo — Predict on New Samples (Pipeline)

In [21]:
# Predict from raw X_test — pipeline handles all preprocessing internally
sample_raw = X_test.iloc[:5]

pred_types = le_target.inverse_transform(clf_pipeline.predict(sample_raw))
pred_sevs  = reg_pipeline.predict(sample_raw)
true_types = le_target.inverse_transform(y_clf_test.values[:5])
true_sevs  = y_reg_test.values[:5]

result = pd.DataFrame({
    'True Type'      : true_types,
    'Pred Type'      : pred_types,
    'Match?'         : true_types == pred_types,
    'True Severity'  : true_sevs.round(2),
    'Pred Severity'  : pred_sevs.round(2),
    'Severity Error' : (true_sevs - pred_sevs).round(3)
})
display(result)


,True Type,Pred Type,Match?,True Severity,Pred Severity,Severity Error
0,Flood,Flood,True,74.88,74.459999,0.417
1,Water Clogging,Water Clogging,True,32.79,33.970001,-1.186
2,Heatwave,Heatwave,True,48.17,47.570000,0.601
3,Lightning,Lightning,True,37.78,37.240002,0.541
4,Lightning,Lightning,True,33.07,33.099998,-0.035


## 📋 Fix Log

| # | Issue | Fix |
|---|---|---|
| 1 | Wrong CSV filename | `aegisgrid_final_dataset.csv` |
| 2 | Incomplete `DROP_COLS` | All `*_risk_*`, `*_probability`, `recommended_action` added |
| 3 | Target leakage in features | `flood_probability`, `heat_probability` excluded from X |
| 4 | `LabelEncoder` on nominals | `OneHotEncoder` via `ColumnTransformer` |
| 5 | Binary `is_*` cols as object | Cast to `int` before dtype detection |
| 6 | Unnecessary `StandardScaler` | Removed (tree models are scale-invariant) |
| 7 | sklearn substitute | Real `xgboost` library throughout |
| 8 | No class imbalance handling | `sample_weight='balanced'` passed to `fit()` — no SMOTE |
| 9 | Truncated visualization | Full 8-plot dashboard restored |
| 10 | SHAP used before `import shap` | `import shap` added; explainer defined before use |
| 11 | SHAP feature names `f0,f1...` | Real `all_feature_names` list used |
| 12 | Pipeline double-transform | Pipelines fit fresh models on raw `X_train` |
| 13 | No variance filter | `VarianceThreshold(0.01)` removes near-constant features |
| 14 | CV ignores sample weights | `fit_params={'sample_weight':...}` added to `cross_val_score` |
